# Pneumonia Detection — Streamlit App

Runs the trained EfficientNetV2-S + MixStyle + SWAD model as a Streamlit web app.

## Run order
Cell 1 → Cell 2 → Cell 3 → open the ngrok URL

In [1]:
# CELL 1 — Install dependencies
!pip install -q streamlit pyngrok opencv-python-headless pillow
!pip install -q tensorflow albumentations

import os
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 88.4 MB/s eta 0:00:00
Done


In [8]:
# CELL 2 — Mount Drive and write the app file
from google.colab import drive
drive.mount('/content/drive')

APP_CODE = """\
import streamlit as st
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cmx
import tensorflow as tf
from PIL import Image
import io
import os
import glob

st.set_page_config(page_title="Pneumonia Detection", page_icon="X", layout="wide", initial_sidebar_state="expanded")

st.markdown(\"\"\"
<style>
.main-title  {font-size:2rem;font-weight:600;color:#1F3864;margin-bottom:.1rem}
.sub-title   {font-size:.95rem;color:#888;margin-bottom:1.5rem}
.result-box  {padding:1.25rem;border-radius:12px;text-align:center;margin:.75rem 0}
.pneumonia   {background:#FCEBEB;border:2px solid #E24B4A}
.normal      {background:#EAF3DE;border:2px solid #639922}
.uncertain   {background:#FAEEDA;border:2px solid #BA7517}
.result-icon {font-size:2rem;margin-bottom:.4rem}
.result-text {font-size:1.2rem;font-weight:600}
.mcard       {background:#F8F9FA;border-radius:8px;padding:.9rem 1rem;text-align:center;border:1px solid #E9ECEF}
.mval        {font-size:1.6rem;font-weight:600;color:#1F3864}
.mlbl        {font-size:.75rem;color:#888;margin-top:.15rem}
.info-box    {background:#E6F1FB;border-left:4px solid #185FA5;padding:.6rem 1rem;border-radius:0 8px 8px 0;font-size:.875rem;color:#0C447C;margin:.5rem 0}
.warn-box    {background:#FAEEDA;border-left:4px solid #BA7517;padding:.6rem 1rem;border-radius:0 8px 8px 0;font-size:.875rem;color:#633806;margin:.5rem 0}
.danger-box  {background:#FCEBEB;border-left:4px solid #E24B4A;padding:.6rem 1rem;border-radius:0 8px 8px 0;font-size:.875rem;color:#A32D2D;margin:.5rem 0}
.success-box {background:#EAF3DE;border-left:4px solid #639922;padding:.6rem 1rem;border-radius:0 8px 8px 0;font-size:.875rem;color:#27500A;margin:.5rem 0}
</style>
\"\"\", unsafe_allow_html=True)

IMG_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
MC_PASSES = 20
BASE = '/content/drive/MyDrive/FYP_Pneumonia_DG'

class RandConv(tf.keras.layers.Layer):
    def __init__(self, p=0.5, **kwargs):
        super().__init__(**kwargs); self.p = p
    def call(self, x, training=None):
        if not training: return x
        def apply_rc():
            kernel = tf.random.normal([1, 1, 3, 3])
            kernel = kernel / (tf.norm(kernel) + 1e-7)
            out    = tf.nn.conv2d(x, kernel, strides=[1,1,1,1], padding='SAME')
            out    = tf.clip_by_value(out, -10.0, 10.0)
            lam    = tf.random.uniform([], 0.0, 1.0)
            return lam * x + (1.0 - lam) * out
        return tf.cond(tf.random.uniform([]) < self.p, apply_rc, lambda: x)
    def get_config(self):
        c = super().get_config(); c.update({'p': self.p}); return c

class MixStyle(tf.keras.layers.Layer):
    def __init__(self, alpha=0.1, p=0.5, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.alpha=alpha; self.p=p; self.eps=eps
    def _mix(self, x):
        x=tf.cast(x,tf.float32); B=tf.shape(x)[0]
        mu =tf.reduce_mean(x, axis=[1,2], keepdims=True)
        sig=tf.sqrt(tf.math.reduce_variance(x, axis=[1,2], keepdims=True)+self.eps)
        xn =(x-mu)/sig
        g1 =-tf.math.log(tf.random.uniform([B,1,1,1],1e-8,1.))/self.alpha
        g2 =-tf.math.log(tf.random.uniform([B,1,1,1],1e-8,1.))/self.alpha
        lam=g1/(g1+g2); perm=tf.random.shuffle(tf.range(B))
        return xn*(lam*sig+(1-lam)*tf.gather(sig,perm))+(lam*mu+(1-lam)*tf.gather(mu,perm))
    def call(self, x, training=None):
        if not training: return x
        return tf.cond(tf.random.uniform([])<self.p,
                       lambda:self._mix(x), lambda:tf.cast(x,tf.float32))
    def get_config(self):
        c=super().get_config()
        c.update({'alpha':self.alpha,'p':self.p,'eps':self.eps}); return c

class FocalBCE(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.75, ls=0.05, **kw):
        super().__init__(**kw)
        self.gamma=gamma; self.alpha=alpha; self.ls=ls
    def call(self, yt, yp):
        yt=tf.cast(yt,tf.float32); yp=tf.cast(yp,tf.float32)
        if len(yp.shape)>len(yt.shape): yp=tf.squeeze(yp,-1)
        yt=yt*(1-self.ls)+0.5*self.ls
        yp=tf.clip_by_value(yp,1e-7,1-1e-7)
        bce=-(yt*tf.math.log(yp)+(1-yt)*tf.math.log(1-yp))
        pt =yt*yp+(1-yt)*(1-yp)
        return tf.reduce_mean((yt*self.alpha+(1-yt)*(1-self.alpha))*tf.pow(1-pt,self.gamma)*bce)
    def get_config(self):
        c=super().get_config()
        c.update({'gamma':self.gamma,'alpha':self.alpha,'ls':self.ls}); return c

@st.cache_resource
def load_model():
    candidates = [
        f'{BASE}/checkpoints/v3_fold1_test_rsna_dg_swad.keras',
        f'{BASE}/checkpoints/v3_fold1_test_rsna_dg.keras',
        f'{BASE}/checkpoints/v3_fold2_test_nih_dg_swad.keras',
        f'{BASE}/checkpoints/v3_fold2_test_nih_dg.keras',
    ]
    ckpt = next((c for c in candidates if os.path.exists(c)), None)
    if ckpt is None:
        return None, None, None
    model = tf.keras.models.load_model(
        ckpt, custom_objects={'MixStyle':MixStyle,'FocalBCE':FocalBCE,'RandConv':RandConv}
    )
    bb        = next((l for l in model.layers if hasattr(l,'layers') and len(l.layers)>10), None)
    last_conv = next((l for l in reversed(bb.layers) if isinstance(l, tf.keras.layers.Conv2D)), None)
    feat_extractor = tf.keras.Model(inputs=bb.input, outputs=last_conv.output)
    return model, feat_extractor, os.path.basename(ckpt)

def preprocess(pil_img):
    img = np.array(pil_img.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    cl  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab[:,:,0] = cl.apply(lab[:,:,0])
    img  = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    orig = img.copy()
    inp  = (img.astype(np.float32)/255.0 - MEAN) / STD
    return orig, inp

@tf.function
def _predict_dropout(model, x):
    return model(x, training=True)

def run_mc_dropout(model, inp, n_passes):
    x     = tf.constant(inp[np.newaxis], dtype=tf.float32)
    preds = [float(tf.squeeze(_predict_dropout(model, x)).numpy()) for _ in range(n_passes)]
    return float(np.mean(preds)), float(np.std(preds))

def run_cam(feat_extractor, model, inp):
    x         = inp[np.newaxis].astype(np.float32)
    feat_maps = feat_extractor(x, training=False).numpy()[0]
    pred      = float(model(x, training=False).numpy()[0, 0])
    weights   = np.maximum(np.mean(feat_maps, axis=(0,1)), 0)
    cam       = np.sum(feat_maps * weights[np.newaxis,np.newaxis,:], axis=-1)
    cam       = np.maximum(cam, 0); cam /= cam.max()+1e-7
    cam       = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    return cam, pred

def make_cam_figure(orig, cam, pred):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.patch.set_facecolor('white')
    axes[0].imshow(orig); axes[0].axis('off')
    axes[0].set_title('Original', fontsize=11, fontweight='bold', pad=8)
    heatmap = cmx.jet(cam)[:,:,:3]
    axes[1].imshow(heatmap); axes[1].axis('off')
    axes[1].set_title('Activation map', fontsize=11, fontweight='bold', pad=8)
    overlay = np.clip(orig/255.0*0.55 + heatmap*0.45, 0, 1)
    axes[2].imshow(overlay); axes[2].axis('off')
    col = '#E24B4A' if pred >= 0.5 else '#639922'
    axes[2].set_title(f'Overlay  P = {pred:.3f}', fontsize=11, fontweight='bold', color=col, pad=8)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    buf.seek(0); plt.close()
    return buf

with st.sidebar:
    st.markdown('### Model')
    threshold  = st.slider('Classification threshold', 0.1, 0.9, 0.608, 0.01)
    mc_passes  = st.slider('MC Dropout passes', 5, 30, 20, 5)
    unc_thresh = st.slider('Uncertainty threshold', 0.05, 0.30, 0.15, 0.01)
    st.markdown('---')
    st.markdown('### Results (LODO)')
    st.markdown('| Domain | AUC |\\n|--------|-----|\\n| Kermany | **0.9687** |\\n| RSNA | 0.7983 |\\n| NIH | 0.7452 |')
    st.markdown('---')
    st.caption('EfficientNetV2-S + RandConv + MixStyle x2 + SWAD\\nLGU FYP 2024 — Ali Shameer')

st.markdown('<div class="main-title">Pneumonia Detection</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-title">Cross-hospital domain generalization — EfficientNetV2-S + MixStyle + SWAD</div>', unsafe_allow_html=True)

model, feat_extractor, ckpt_name = load_model()

if model is None:
    st.markdown('<div class="danger-box">No checkpoint found in Drive. Check your path.</div>', unsafe_allow_html=True)
    st.stop()
else:
    st.markdown(f'<div class="success-box">Model loaded: <strong>{ckpt_name}</strong></div>', unsafe_allow_html=True)

st.markdown('---')
uploaded = st.file_uploader('Upload a chest X-ray', type=['jpg','jpeg','png'])

if uploaded is not None:
    pil_img = Image.open(uploaded)
    col1, col2 = st.columns([1, 2])
    with col1:
        st.image(pil_img, caption='Uploaded image', use_container_width=True)
        st.caption(f'{pil_img.size[0]} x {pil_img.size[1]} px')
    with col2:
        with st.spinner('Running inference...'):
            orig, inp = preprocess(pil_img)
            cam, _    = run_cam(feat_extractor, model, inp)
        with st.spinner(f'MC Dropout ({mc_passes} passes)...'):
            pred_mean, pred_std = run_mc_dropout(model, inp, mc_passes)
        is_pneumonia = pred_mean >= threshold
        is_uncertain = pred_std  >= unc_thresh
        if is_uncertain:
            css, icon, label = 'uncertain', 'Warning', 'HIGH UNCERTAINTY — Flag for radiologist review'
        elif is_pneumonia:
            css, icon, label = 'pneumonia', 'Alert', 'PNEUMONIA DETECTED'
        else:
            css, icon, label = 'normal', 'OK', 'NORMAL'
        st.markdown(f'<div class="result-box {css}"><div class="result-icon">{icon}</div><div class="result-text">{label}</div></div>', unsafe_allow_html=True)
        c1,c2,c3,c4 = st.columns(4)
        with c1: st.markdown(f'<div class="mcard"><div class="mval">{pred_mean:.3f}</div><div class="mlbl">P(pneumonia)</div></div>', unsafe_allow_html=True)
        with c2: st.markdown(f'<div class="mcard"><div class="mval">{pred_std:.3f}</div><div class="mlbl">Uncertainty (sigma)</div></div>', unsafe_allow_html=True)
        with c3: st.markdown(f'<div class="mcard"><div class="mval">{threshold:.3f}</div><div class="mlbl">Threshold</div></div>', unsafe_allow_html=True)
        with c4:
            flag = 'YES' if is_uncertain else 'No'
            col  = '#BA7517' if is_uncertain else '#3B6D11'
            st.markdown(f'<div class="mcard"><div class="mval" style="color:{col}">{flag}</div><div class="mlbl">Radiologist review</div></div>', unsafe_allow_html=True)
        if is_uncertain:
            st.markdown('<div class="warn-box">High uncertainty — route to radiologist regardless of predicted class.</div>', unsafe_allow_html=True)
        elif is_pneumonia:
            st.markdown('<div class="danger-box">Pneumonia predicted above threshold. See activation map below.</div>', unsafe_allow_html=True)
        else:
            st.markdown('<div class="success-box">Normal predicted with acceptable confidence.</div>', unsafe_allow_html=True)
    st.markdown('---')
    st.markdown('### Class Activation Map')
    cam_buf = make_cam_figure(orig, cam, pred_mean)
    st.image(cam_buf, use_container_width=True)
    unc_pct   = min(pred_std / 0.30 * 100, 100)
    bar_color = '#E24B4A' if pred_std >= unc_thresh else '#639922'
    st.markdown(f'<div style="background:#F1EFE8;border-radius:6px;height:10px;overflow:hidden;margin:6px 0"><div style="width:{unc_pct:.1f}%;height:100%;background:{bar_color};border-radius:6px"></div></div><div style="font-size:.8rem;color:#888">Uncertainty sigma = {pred_std:.4f} (threshold = {unc_thresh:.2f}) — {"Flag for review" if pred_std >= unc_thresh else "Confidence acceptable"}</div>', unsafe_allow_html=True)
    st.markdown('---')
    cam_buf.seek(0)
    st.download_button('Download CAM', cam_buf, f'cam_{uploaded.name}', 'image/png')
else:
    st.markdown('<div class="info-box">Upload a chest X-ray above to begin.</div>', unsafe_allow_html=True)
"""

with open('/content/app.py', 'w') as f:
    f.write(APP_CODE)

print('app.py written successfully')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
app.py written successfully


In [ ]:
# CELL 3 — Launch Streamlit + ngrok
import subprocess, threading, time, sys, os

NGROK_TOKEN = "Token"  # <-- paste your token

# Install if needed
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)

from pyngrok import ngrok, conf

# Set token
conf.get_default().auth_token = NGROK_TOKEN

# Kill anything already running on 8501
ngrok.kill()
subprocess.run(['fuser', '-k', '8501/tcp'], capture_output=True)

# Start streamlit in background thread
proc = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.fileWatcherType', 'none',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for streamlit to be ready
print('Waiting for Streamlit to start...')
time.sleep(12)

# Check it started
try:
    import urllib.request
    urllib.request.urlopen('http://localhost:8501', timeout=5)
    print('Streamlit is running.')
except Exception as e:
    print(f'Streamlit check: {e}')
    # Print any startup errors
    out = proc.stdout.read1(2000) if hasattr(proc.stdout, 'read1') else b''
    err = proc.stderr.read1(2000) if hasattr(proc.stderr, 'read1') else b''
    if out: print('stdout:', out.decode())
    if err: print('stderr:', err.decode())

# Open ngrok tunnel
try:
    public_url = ngrok.connect(8501)
    print(f'\n========================================')
    print(f'App is live at:')
    print(f'{public_url}')
    print(f'========================================')
    print('Open the URL above in any browser.')
    print('Keep this cell running — closing it stops the app.')
except Exception as e:
    print(f'ngrok error: {e}')
    print('Try running this cell again.')

Waiting for Streamlit to start...
Streamlit is running.

App is live at:
NgrokTunnel: "https://antennae-unboxed-parish.ngrok-free.dev" -> "http://localhost:8501"
Open the URL above in any browser.
Keep this cell running — closing it stops the app.
